# Interactive Data Visualization with Controls

This notebook creates an interactive form with data controls to manipulate visualizations in real-time using ipywidgets and plotly.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

## Generate Sample Data

First, let's create some sample data to work with - sales data across different regions and time periods.

In [3]:
# Generate sample sales data
np.random.seed(42)
dates = pd.date_range('2020-01-01', '2023-12-31', freq='M')
regions = ['North', 'South', 'East', 'West', 'Central']
products = ['Product A', 'Product B', 'Product C', 'Product D']

# Create comprehensive dataset
data = []
for date in dates:
    for region in regions:
        for product in products:
            sales = np.random.normal(1000, 300) + np.random.normal(0, 100)
            profit = sales * np.random.uniform(0.1, 0.3)
            quantity = int(sales / np.random.uniform(20, 80))
            data.append({
                'date': date,
                'region': region,
                'product': product,
                'sales': max(0, sales),
                'profit': max(0, profit),
                'quantity': max(1, quantity)
            })

df = pd.DataFrame(data)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (960, 9)


,date,region,product,sales,profit,quantity,year,month,quarter
0,2020-01-31,North,Product A,1135.187816,279.708902,20,2020,1,1
1,2020-01-31,North,Product B,906.340292,101.162733,12,2020,1,1
2,2020-01-31,North,Product C,1550.507318,161.434014,19,2020,1,1
3,2020-01-31,North,Product D,913.413689,124.557652,29,2020,1,1
4,2020-01-31,South,Product A,881.260657,164.257296,23,2020,1,1


## Create Interactive Controls

Now let's create various widgets to control our visualization parameters.

In [4]:
# Create interactive widgets
style = {'description_width': '120px'}
layout = widgets.Layout(width='300px')

# Data filtering controls
region_dropdown = widgets.SelectMultiple(
    options=regions,
    value=regions[:3],
    description='Regions:',
    style=style,
    layout=layout
)

product_dropdown = widgets.SelectMultiple(
    options=products,
    value=products[:2],
    description='Products:',
    style=style,
    layout=layout
)

year_slider = widgets.SelectionRangeSlider(
    options=sorted(df['year'].unique()),
    index=(0, len(df['year'].unique())-1),
    description='Year Range:',
    style=style,
    layout=layout
)

# Visualization controls
chart_type = widgets.Dropdown(
    options=['Line Chart', 'Bar Chart', 'Scatter Plot', 'Box Plot', 'Heatmap'],
    value='Line Chart',
    description='Chart Type:',
    style=style,
    layout=layout
)

metric_dropdown = widgets.Dropdown(
    options=['sales', 'profit', 'quantity'],
    value='sales',
    description='Metric:',
    style=style,
    layout=layout
)

group_by = widgets.Dropdown(
    options=['region', 'product', 'year', 'quarter'],
    value='region',
    description='Group By:',
    style=style,
    layout=layout
)

# Appearance controls
color_scheme = widgets.Dropdown(
    options=['plotly', 'viridis', 'plasma', 'blues', 'reds'],
    value='plotly',
    description='Color Scheme:',
    style=style,
    layout=layout
)

show_trend = widgets.Checkbox(
    value=False,
    description='Show Trend Line',
    style=style
)

# Display controls in organized layout
filter_box = widgets.VBox([
    widgets.HTML("<h3>Data Filters</h3>"),
    region_dropdown,
    product_dropdown,
    year_slider
])

visualization_box = widgets.VBox([
    widgets.HTML("<h3>Visualization Settings</h3>"),
    chart_type,
    metric_dropdown,
    group_by
])

appearance_box = widgets.VBox([
    widgets.HTML("<h3>Appearance</h3>"),
    color_scheme,
    show_trend
])

control_panel = widgets.HBox([filter_box, visualization_box, appearance_box])
display(control_panel)

## Interactive Visualization Function

Create a function that updates the visualization based on the control settings.

In [5]:
def create_visualization(regions_selected, products_selected, year_range, 
                        chart_type_selected, metric_selected, group_by_selected,
                        color_scheme_selected, show_trend_line):
    
    # Filter data based on selections
    filtered_df = df[
        (df['region'].isin(regions_selected)) &
        (df['product'].isin(products_selected)) &
        (df['year'] >= year_range[0]) &
        (df['year'] <= year_range[1])
    ].copy()
    
    if filtered_df.empty:
        fig = go.Figure()
        fig.add_annotation(
            text="No data available for selected filters",
            xref="paper", yref="paper",
            x=0.5, y=0.5, xanchor='center', yanchor='middle',
            font=dict(size=20)
        )
        return fig
    
    # Group data based on selection
    if chart_type_selected == 'Heatmap':
        # Special handling for heatmap
        pivot_data = filtered_df.groupby(['region', 'product'])[metric_selected].sum().reset_index()
        pivot_table = pivot_data.pivot(index='region', columns='product', values=metric_selected)
        
        fig = px.imshow(
            pivot_table,
            title=f'{metric_selected.title()} by Region and Product',
            color_continuous_scale=color_scheme_selected,
            aspect='auto'
        )
    else:
        # Group data for other chart types
        if group_by_selected == 'year':
            grouped_df = filtered_df.groupby(['year', 'region' if len(regions_selected) > 1 else 'product'])[metric_selected].sum().reset_index()
            x_col = 'year'
            color_col = 'region' if len(regions_selected) > 1 else 'product'
        elif group_by_selected == 'quarter':
            filtered_df['year_quarter'] = filtered_df['year'].astype(str) + '-Q' + filtered_df['quarter'].astype(str)
            grouped_df = filtered_df.groupby(['year_quarter', 'region' if len(regions_selected) > 1 else 'product'])[metric_selected].sum().reset_index()
            x_col = 'year_quarter'
            color_col = 'region' if len(regions_selected) > 1 else 'product'
        else:
            grouped_df = filtered_df.groupby([group_by_selected])[metric_selected].sum().reset_index()
            x_col = group_by_selected
            color_col = None
        
        # Create visualization based on chart type
        if chart_type_selected == 'Line Chart':
            if color_col:
                fig = px.line(
                    grouped_df, x=x_col, y=metric_selected, color=color_col,
                    title=f'{metric_selected.title()} Over Time by {color_col.title()}',
                    color_discrete_sequence=px.colors.qualitative.Set1
                )
            else:
                fig = px.line(
                    grouped_df, x=x_col, y=metric_selected,
                    title=f'{metric_selected.title()} by {group_by_selected.title()}'
                )
                
        elif chart_type_selected == 'Bar Chart':
            if color_col:
                fig = px.bar(
                    grouped_df, x=x_col, y=metric_selected, color=color_col,
                    title=f'{metric_selected.title()} by {group_by_selected.title()}',
                    color_discrete_sequence=px.colors.qualitative.Set1
                )
            else:
                fig = px.bar(
                    grouped_df, x=x_col, y=metric_selected,
                    title=f'{metric_selected.title()} by {group_by_selected.title()}'
                )
                
        elif chart_type_selected == 'Scatter Plot':
            # Use original filtered data for scatter plot to show individual points
            fig = px.scatter(
                filtered_df, x='sales', y='profit', color='region', size='quantity',
                title='Sales vs Profit (Size = Quantity)',
                color_discrete_sequence=px.colors.qualitative.Set1
            )
            
        elif chart_type_selected == 'Box Plot':
            fig = px.box(
                filtered_df, x=group_by_selected, y=metric_selected,
                title=f'{metric_selected.title()} Distribution by {group_by_selected.title()}'
            )
    
    # Add trend line if requested and applicable
    if show_trend_line and chart_type_selected in ['Line Chart', 'Scatter Plot'] and not filtered_df.empty:
        if chart_type_selected == 'Scatter Plot':
            # Add trend line to scatter plot
            z = np.polyfit(filtered_df['sales'], filtered_df['profit'], 1)
            p = np.poly1d(z)
            fig.add_traces(go.Scatter(x=filtered_df['sales'], y=p(filtered_df['sales']),
                                    mode="lines", name='Trend Line',
                                    line=dict(color='red', dash='dash')))
    
    # Update layout
    fig.update_layout(
        height=500,
        showlegend=True,
        template='plotly_white'
    )
    
    return fig

# Create output widget for the plot
output = widgets.Output()

def update_plot(*args):
    with output:
        clear_output(wait=True)
        fig = create_visualization(
            region_dropdown.value,
            product_dropdown.value, 
            year_slider.value,
            chart_type.value,
            metric_dropdown.value,
            group_by.value,
            color_scheme.value,
            show_trend.value
        )
        fig.show()

# Connect widgets to update function
region_dropdown.observe(update_plot, 'value')
product_dropdown.observe(update_plot, 'value')
year_slider.observe(update_plot, 'value')
chart_type.observe(update_plot, 'value')
metric_dropdown.observe(update_plot, 'value')
group_by.observe(update_plot, 'value')
color_scheme.observe(update_plot, 'value')
show_trend.observe(update_plot, 'value')

# Display the output
display(output)

# Generate initial plot
update_plot()

Output()

## Summary Statistics Panel

Add a panel showing key statistics that update with the filters.

In [6]:
# Create summary statistics widgets
summary_output = widgets.Output()

def update_summary(*args):
    with summary_output:
        clear_output(wait=True)
        
        # Filter data based on current selections
        filtered_df = df[
            (df['region'].isin(region_dropdown.value)) &
            (df['product'].isin(product_dropdown.value)) &
            (df['year'] >= year_slider.value[0]) &
            (df['year'] <= year_slider.value[1])
        ].copy()
        
        if not filtered_df.empty:
            total_sales = filtered_df['sales'].sum()
            total_profit = filtered_df['profit'].sum()
            total_quantity = filtered_df['quantity'].sum()
            avg_profit_margin = (total_profit / total_sales * 100) if total_sales > 0 else 0
            
            # Create summary display
            print("📊 SUMMARY STATISTICS")
            print("=" * 50)
            print(f"Total Sales:        ${total_sales:,.2f}")
            print(f"Total Profit:       ${total_profit:,.2f}")
            print(f"Total Quantity:     {total_quantity:,} units")
            print(f"Profit Margin:      {avg_profit_margin:.1f}%")
            print(f"Records Filtered:   {len(filtered_df):,}")
            print("=" * 50)
            
            # Top performers
            top_region = filtered_df.groupby('region')['sales'].sum().idxmax()
            top_product = filtered_df.groupby('product')['sales'].sum().idxmax()
            
            print(f"Top Region:         {top_region}")
            print(f"Top Product:        {top_product}")
        else:
            print("No data available for current filters.")

# Connect summary to widget updates
region_dropdown.observe(update_summary, 'value')
product_dropdown.observe(update_summary, 'value')
year_slider.observe(update_summary, 'value')

# Display summary
print("\n📈 Real-time Summary Statistics:")
display(summary_output)
update_summary()


📈 Real-time Summary Statistics:


Output()

## Usage Instructions

### Available Controls:

1. **Data Filters:**
   - **Regions**: Select multiple regions to include in analysis
   - **Products**: Choose which products to analyze
   - **Year Range**: Set the time period for analysis

2. **Visualization Settings:**
   - **Chart Type**: Choose from Line Chart, Bar Chart, Scatter Plot, Box Plot, or Heatmap
   - **Metric**: Select sales, profit, or quantity to visualize
   - **Group By**: Group data by region, product, year, or quarter

3. **Appearance:**
   - **Color Scheme**: Choose from different color palettes
   - **Show Trend Line**: Add trend lines to applicable charts

### Tips:
- Try different combinations of chart types and groupings
- Use the scatter plot to explore relationships between sales and profit
- The heatmap is great for comparing performance across regions and products
- Summary statistics update automatically as you change filters

### Next Steps:
- Add more sophisticated filtering options
- Include additional chart types or customization options
- Export functionality for charts and data
- Add statistical analysis features